In [6]:
import pickle
import json
from pathlib import Path
import numpy as np
import pandas as pd

from neurosym.examples.near.metrics_torch_ecg import compute_torch_ecg_classification_metrics


In [7]:
from pathlib import Path

TASK_LABEL_MODE = "multi"
assert TASK_LABEL_MODE in {"single", "multi"}


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    markers = ("pyproject.toml", ".git", "neurosym")
    for path in (start, *start.parents):
        if any((path / m).exists() for m in markers):
            return path
    return start


def infer_label_mode_from_name(name: str):
    lower_name = name.lower()
    if "_multi" in lower_name or lower_name.endswith("multi"):
        return "multi"
    if "_single" in lower_name or lower_name.endswith("single"):
        return "single"
    return None


def infer_near_label_mode(file_path: Path):
    summary_path = file_path.with_name(f"{file_path.stem}_summary.json")
    if summary_path.exists():
        try:
            with open(summary_path, 'r', encoding='utf-8') as f:
                summary_data = json.load(f)
            mode = summary_data.get("label_mode")
            if mode in {"single", "multi"}:
                return mode
        except (OSError, json.JSONDecodeError):
            pass

    inferred = infer_label_mode_from_name(file_path.stem)
    if inferred in {"single", "multi"}:
        return inferred

    legacy_single_files = {
        "reproduction_simple",
        "reproduction_attention",
        "reproduction_attention_drop_eg",
    }
    if file_path.stem in legacy_single_files:
        return "single"

    return None


repo_root = find_repo_root(Path.cwd())
results_dir = repo_root / 'outputs' / 'ecg_results'
print(f"Analyzing ECG task: {TASK_LABEL_MODE}")

# Load NEAR result pickles for the selected task.
neurosym_results = {}
near_stems = [
    'reproduction_simple',
    'reproduction_attention',
    'reproduction_attention_drop_eg',
]
near_candidates = []
for stem in near_stems:
    near_candidates.append(results_dir / f'{stem}.pkl')
    near_candidates.append(results_dir / f'{stem}_{TASK_LABEL_MODE}.pkl')
    near_candidates.extend(sorted(results_dir.glob(f'{stem}_{TASK_LABEL_MODE}*.pkl')))

near_files = []
seen_near = set()
for candidate in near_candidates:
    if candidate.exists() and candidate not in seen_near:
        near_files.append(candidate)
        seen_near.add(candidate)

for file_path in near_files:
    detected_mode = infer_near_label_mode(file_path)
    if detected_mode != TASK_LABEL_MODE:
        continue

    with open(file_path, 'rb') as f:
        results = pickle.load(f)

    for i, result in enumerate(results):
        key = f"neurosym_{file_path.stem}_{i:03d}"
        if isinstance(result, dict):
            result = dict(result)
            result.setdefault("label_mode", detected_mode)
        neurosym_results[key] = result
    print(f"  Loaded {len(results)} programs from {file_path.stem}")

if not neurosym_results:
    print(f"Warning: no NEAR result pickles found for label_mode={TASK_LABEL_MODE}")

# Load baseline JSON results for the selected task.
baseline_results = {}
baseline_files = sorted(results_dir.glob('baseline_*.json'))

for bf in baseline_files:
    with open(bf, 'r', encoding='utf-8') as f:
        data = json.load(f)

    file_label_mode = data.get('label_mode') if isinstance(data, dict) else None
    if file_label_mode not in {'single', 'multi'}:
        file_label_mode = infer_label_mode_from_name(bf.stem)

    if file_label_mode != TASK_LABEL_MODE:
        continue

    # torch-ecg baselines store one JSON file with per-model reports.
    if isinstance(data, dict) and isinstance(data.get('reports'), list):
        for report in data['reports']:
            model_name = report.get('model', 'unknown_model')
            key = f"baseline_{bf.stem}_{model_name}"
            baseline_results[key] = {
                'report': report.get('test_metrics', {}),
                'time': report.get('train_time_sec', 0.0),
                'label_mode': file_label_mode,
            }
    else:
        key = f"baseline_{bf.stem}"
        if isinstance(data, dict):
            entry = dict(data)
            entry['label_mode'] = file_label_mode
            baseline_results[key] = entry

if baseline_results:
    print(f"  Loaded {len(baseline_results)} baseline results")
else:
    print(f"Warning: no baseline results found for label_mode={TASK_LABEL_MODE}")


Analyzing ECG task: multi
  Loaded 5 baseline results


In [8]:
all_results = {}
all_results.update(neurosym_results)
all_results.update(baseline_results)


def _as_float(value, default=np.nan):
    try:
        return float(value)
    except (TypeError, ValueError):
        return float(default)


# Create comparison dataframe
table_data = []

for name, result in all_results.items():
    if not isinstance(result, dict):
        continue

    result_label_mode = result.get('label_mode')
    if result_label_mode in {'single', 'multi'} and result_label_mode != TASK_LABEL_MODE:
        continue

    report = result.get("report", {}) if isinstance(result, dict) else {}

    macro_precision = np.nan
    macro_recall = np.nan
    macro_f1 = np.nan
    macro_accuracy = np.nan
    macro_auroc = np.nan
    macro_auprc = np.nan
    support = np.nan

    if isinstance(report, dict):
        macro_precision = _as_float(report.get("macro_prec", report.get("precision", np.nan)))
        macro_recall = _as_float(report.get("macro_sens", report.get("recall", np.nan)))
        macro_f1 = _as_float(report.get("macro_f1", report.get("f1_score", np.nan)))
        macro_accuracy = _as_float(report.get("macro_acc", report.get("hamming_accuracy", np.nan)))
        macro_auroc = _as_float(report.get("macro_auroc", np.nan))
        macro_auprc = _as_float(report.get("macro_auprc", np.nan))

        if not np.isfinite(macro_precision) or not np.isfinite(macro_recall) or not np.isfinite(macro_f1):
            macro_avg = {}
            if "report" in report and isinstance(report["report"], dict):
                macro_avg = report["report"].get("macro avg", {})
            elif "macro avg" in report and isinstance(report["macro avg"], dict):
                macro_avg = report["macro avg"]
            if macro_avg:
                macro_precision = _as_float(macro_avg.get("precision", macro_precision))
                macro_recall = _as_float(macro_avg.get("recall", macro_recall))
                macro_f1 = _as_float(macro_avg.get("f1-score", macro_f1))
                support = _as_float(macro_avg.get("support", support))

    if not np.isfinite(macro_f1):
        if "pred_vals" not in result or "true_vals" not in result:
            continue
        pred_vals = np.asarray(result["pred_vals"])
        true_vals = np.asarray(result["true_vals"])
        fallback_label_mode = result.get("label_mode", TASK_LABEL_MODE)
        if fallback_label_mode not in {"single", "multi"}:
            fallback_label_mode = "multi" if true_vals.ndim > 1 and true_vals.shape[-1] > 1 else "single"
        fallback = compute_torch_ecg_classification_metrics(
            pred_vals, true_vals, label_mode=fallback_label_mode
        )
        macro_precision = _as_float(fallback.get("macro_prec", macro_precision))
        macro_recall = _as_float(fallback.get("macro_sens", macro_recall))
        macro_f1 = _as_float(fallback.get("macro_f1", macro_f1))
        macro_accuracy = _as_float(fallback.get("macro_acc", macro_accuracy))
        macro_auroc = _as_float(fallback.get("macro_auroc", macro_auroc))
        macro_auprc = _as_float(fallback.get("macro_auprc", macro_auprc))
        support = _as_float(
            fallback.get("report", {}).get("macro avg", {}).get("support", support)
        )

    runtime = result.get("time", result.get("train_time", result.get("train_time_sec", 0.0)))
    row = {
        "task": TASK_LABEL_MODE,
        "experiment": name,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "macro_accuracy": macro_accuracy,
        "macro_auroc": macro_auroc,
        "macro_auprc": macro_auprc,
        "support": support,
        "time": runtime,
    }
    table_data.append(row)

df = pd.DataFrame(table_data)
if not df.empty:
    df = df.sort_values(["macro_f1", "macro_accuracy"], ascending=[False, False])

print("\n" + "=" * 80)
print(f"RESULTS COMPARISON ({TASK_LABEL_MODE})")
print("=" * 80)
print(df.to_string(index=False))
print("=" * 80)



RESULTS COMPARISON (multi)
 task                                              experiment  macro_precision  macro_recall  macro_f1  macro_accuracy  macro_auroc  macro_auprc  support      time
multi baseline_baseline_torch_ecg_multi_ecg_crnn_multi_scopic         0.359945      0.150252  0.236200        0.865687     0.677134     0.250279      NaN 40.647166
multi              baseline_baseline_tree_decision_tree_multi         0.142657      0.380270  0.200544        0.650624     0.493319     0.125746      NaN  7.764599
multi              baseline_baseline_tree_random_forest_multi         0.425901      0.172171  0.194721        0.874839     0.699950     0.305670      NaN 39.705552
multi     baseline_baseline_torch_ecg_multi_multi_scopic_head         0.426081      0.121399  0.158408        0.864610     0.684124     0.247061      NaN 38.067763
multi                              baseline_baseline_nn_multi         0.000000      0.000000  0.000000        0.881675     0.495252     0.128370      Na

In [9]:
print("\nSUMMARY STATISTICS")
print("=" * 80)
print(f"Task: {TASK_LABEL_MODE}")
print(f"Total experiments: {len(df)}")
if not df.empty:
    print(f"\nBest Macro F1: {df['macro_f1'].max():.6f}")
    print(f"  Experiment: {df.loc[df['macro_f1'].idxmax(), 'experiment']}")
    print(f"\nBest Macro Accuracy: {df['macro_accuracy'].max():.6f}")
    print(f"  Experiment: {df.loc[df['macro_accuracy'].idxmax(), 'experiment']}")
    print(f"\nAverage Macro F1: {df['macro_f1'].mean():.6f}")
    print(f"Average Macro Accuracy: {df['macro_accuracy'].mean():.6f}")
    print(f"Average training time: {df['time'].mean():.2f} seconds")
print("=" * 80)



SUMMARY STATISTICS
Task: multi
Total experiments: 5

Best Macro F1: 0.236200
  Experiment: baseline_baseline_torch_ecg_multi_ecg_crnn_multi_scopic

Best Macro Accuracy: 0.881675
  Experiment: baseline_baseline_nn_multi

Average Macro F1: 0.157975
Average Macro Accuracy: 0.827487
Average training time: 26.00 seconds


In [10]:
# Save task-scoped comparison outputs
output_dir = repo_root / 'outputs' / 'ecg_results'
output_dir.mkdir(parents=True, exist_ok=True)

suffix = '' if TASK_LABEL_MODE == 'single' else f'_{TASK_LABEL_MODE}'
csv_path = output_dir / f'comparison{suffix}.csv'
df.to_csv(csv_path, index=False)
print(f"Saved CSV to: {csv_path}")

md_path = output_dir / f'comparison{suffix}.md'
with open(md_path, 'w', encoding='utf-8') as f:
    f.write(f'# ECG NEAR Results Comparison ({TASK_LABEL_MODE})\n\n')
    f.write(df.to_markdown(index=False))
    f.write('\n')
print(f"Saved Markdown to: {md_path}")


Saved CSV to: /home/asehgal/neurosym-lib/outputs/ecg_results/comparison_multi.csv
Saved Markdown to: /home/asehgal/neurosym-lib/outputs/ecg_results/comparison_multi.md
